# 08 · `gl_engine/interp/values.py`

## What this file is for

The runtime value model: five types, and **a null that is not zero**.

ISO's rules are typed, and the types matter more than they would in most programs, because a wrong coercion here doesn't crash — it produces a plausible premium. The two decisions this file makes and defends are: an empty cell is `None` and never `0`, and a condition must be a genuine boolean, because *"silently treating `0` as false is how a rate of zero becomes a control-flow decision."*

**Depends on:** [`02-errors`](02-errors.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.interp import values

for name, obj in vars(values).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != values.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Coerce some text the way ISO declares it.

In [ ]:
from gl_engine.interp import values as V

for raw, typ in [("1000", "decimal"), ("1000", "integer"),
                 ("1000", "long"), ("GA", "string")]:
    got = V.coerce(raw, typ)
    print(f"{raw!r:<12} as {typ:<8} -> {got!r:<28} {type(got).__name__}")

## The interesting case

### Empty is null, and null is not zero

In [ ]:
empty = V.coerce("", "decimal")
zero  = V.coerce("0", "decimal")

print("empty cell ->", repr(empty))
print("a filed 0  ->", repr(zero))
print("same?      ->", empty == zero)

That distinction runs all the way up. An empty cell means *ISO filed nothing here*; a `0` means *ISO filed zero*. In a rating engine those have completely different consequences, and the difference is invisible once they've been collapsed.

It is also the seam where OI-88 lives: a null arriving through arithmetic should yield null and let the countrywide fallback happen, not refuse.

### A condition must be an actual boolean

Most languages would let you write `if 0:`. This one refuses, on purpose.

In [ ]:
for v in [True, False, 0, 1, "", "false", None]:
    try:
        print(f"truthy({v!r:<8}) -> {V.truthy(v)}")
    except V.InterpretError as e:
        print(f"truthy({v!r:<8}) -> REFUSED: {str(e).split('--')[0].strip()}")

### Equality is by value, not by text

`1` and `1.0` are the same number even though they are different strings — and ISO's tables are full of both.

In [ ]:
a = V.coerce("1", "decimal")
b = V.coerce("1.0", "decimal")
print(f"{a!r} == {b!r} ->", V.equal(a, b))
print("text equal? ->", "1" == "1.0")
print()
print("to_text(1.0)   ->", repr(V.to_text(b)))
print("to_decimal('2')->", repr(V.to_decimal("2")))

## What it refuses

A value that cannot be the declared type raises, rather than becoming a default.

In [ ]:
for raw, typ in [("banana", "decimal"), ("1,000", "decimal"), ("2.5", "integer")]:
    try:
        print(f"{raw!r} as {typ}:", V.coerce(raw, typ))
    except V.InterpretError as e:
        print(f"{raw!r} as {typ}: REFUSED -- {str(e).split('--')[0].strip()}")

`"1,000"` is the instructive one. A comma-formatted number looks harmless, and a lenient parser would read it as `1`. Refusing means a submission that formats its numbers differently fails loudly at the boundary instead of pricing at a thousandth of the intended exposure.

## Try it yourself

1. `Multi` and `flatten` handle a value that is several values. Where in ISO's language does that arise?
2. `compare_key` exists so values of different types can be ordered. What does it do with a null?
3. Find where in the engine `truthy` is called. What is the trace of a condition that refuses?

In [ ]:
# your turn